In [14]:
import pandas as pd
import requests
import ast
import time
import json
from collections import defaultdict

# =========================================================
# CONFIG
# =========================================================

INPUT_CSV = "experiment_onlyentityresults_mini3.csv"

OUTPUT_CSV = "state_prediction_results7.csv"

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

HEADERS = {
    "User-Agent": "geo-location-research"
}

# =========================================================
# CACHE
# =========================================================

osm_cache = {}

US_STATES = {

    "alabama",
    "alaska",
    "arizona",
    "arkansas",
    "california",
    "colorado",
    "connecticut",
    "delaware",
    "florida",
    "georgia",
    "hawaii",
    "idaho",
    "illinois",
    "indiana",
    "iowa",
    "kansas",
    "kentucky",
    "louisiana",
    "maine",
    "maryland",
    "massachusetts",
    "michigan",
    "minnesota",
    "mississippi",
    "missouri",
    "montana",
    "nebraska",
    "nevada",
    "new hampshire",
    "new jersey",
    "new mexico",
    "new york",
    "north carolina",
    "north dakota",
    "ohio",
    "oklahoma",
    "oregon",
    "pennsylvania",
    "rhode island",
    "south carolina",
    "south dakota",
    "tennessee",
    "texas",
    "utah",
    "vermont",
    "virginia",
    "washington",
    "west virginia",
    "wisconsin",
    "wyoming"
}
# =========================================================
# OSM SEARCH
# =========================================================

def search_osm(query, limit=5):

    cache_key = f"{query}_{limit}"

    # -----------------------------------------------------
    # cache
    # -----------------------------------------------------

    if cache_key in osm_cache:
        return osm_cache[cache_key]

    params = {
        "q": query,
        "format": "jsonv2",
        "addressdetails": 1,
        "limit": limit,
        "countrycodes": "us"
    }

    try:

        response = requests.get(
            NOMINATIM_URL,
            params=params,
            headers=HEADERS,
            timeout=10
        )

        # -------------------------------------------------
        # invalid status
        # -------------------------------------------------

        if response.status_code != 200:

            print("OSM STATUS ERROR:", response.status_code)

            return []

        # -------------------------------------------------
        # parse json
        # -------------------------------------------------

        results = response.json()

        if not isinstance(results, list):
            return []

        # -------------------------------------------------
        # cache save
        # -------------------------------------------------

        osm_cache[cache_key] = results

        # -------------------------------------------------
        # avoid rate limit
        # -------------------------------------------------

        time.sleep(1)

        return results

    except Exception as e:

        print("OSM ERROR:", query, e)

        return []

# =========================================================
# ADDRESS INFO
# =========================================================

def extract_address_info(item):

    if not isinstance(item, dict):
        return {}

    address = item.get("address", {})

    if not isinstance(address, dict):
        return {}

    city = (
        address.get("city")
        or address.get("town")
        or address.get("village")
        or address.get("municipality")
    )

    county = address.get("county")

    state = address.get("state")

    return {
        "city": city,
        "county": county,
        "state": state
    }

# =========================================================
# PARSE LLM ANALYSIS
# llm_analysis:
# {'entities': ['Colorado', 'High Park']}
# =========================================================

def parse_llm_analysis(llm_str):

    try:

        if pd.isna(llm_str):
            return []

        data = ast.literal_eval(str(llm_str))

        entities = data.get("entities", [])

        parsed_entities = []

        for entity in entities:

            if isinstance(entity, str):

                entity = entity.strip()

                if entity != "":
                    parsed_entities.append(entity)

        return parsed_entities

    except Exception as e:

        print("LLM PARSE ERROR:", e)

        return []

# =========================================================
# USER LOCATION PARSE
# =========================================================

def parse_user_location(user_location):

    if user_location is None:
        return []

    if pd.isna(user_location):
        return []

    results = search_osm(user_location, limit=5)

    if not isinstance(results, list):
        return []

    user_states = []

    for item in results:

        info = extract_address_info(item)

        state = info.get("state")

        if state is not None:

            if state not in user_states:

                user_states.append(state)

    return user_states

# =========================================================
# ENTITY -> STATE CANDIDATES
# =========================================================

def get_entity_state_candidates(query):

    results = search_osm(query, limit=5)

    state_candidates = []

    for item in results:

        info = extract_address_info(item)

        state = info.get("state")

        if state is not None:

            if state not in state_candidates:

                state_candidates.append(state)

    return state_candidates

# =========================================================
# STATE INFERENCE
# =========================================================

def infer_state(entities, user_context):

    state_scores = defaultdict(int)

    debug_info = []

    # =====================================================
    # SINGLE ENTITY
    # =====================================================

    # =====================================================
# SINGLE ENTITY
# =====================================================

    if len(entities) == 1:
    
        entity = entities[0]
    
        # =================================================
        # DIRECT STATE ENTITY
        # =================================================
    
        if entity.lower().strip() in US_STATES:
    
            return {
    
                "status": "resolved",
    
                "predicted_state": entity.title(),
    
                "scores": {
                    entity.title(): 10
                },
    
                "debug": [
    
                    {
                        "entity_type": "state",
    
                        "entity": entity
                    }
                ]
            }
    
        # -------------------------------------------------
        # first retrieval
        # -------------------------------------------------
    
        state_candidates = get_entity_state_candidates(entity)
    
        # -------------------------------------------------
        # no candidates
        # -------------------------------------------------
    
        if len(state_candidates) == 0:
    
            return {
    
                "status": "no_candidate",
    
                "predicted_state": None,
    
                "scores": {},
    
                "debug": []
            }
    
        # =================================================
        # infer entity granularity
        # =================================================
    
        # -------------------------------------------------
        # UNIQUE CANDIDATE
        # -------------------------------------------------
    
        if len(state_candidates) == 1:
    
            candidate_state = state_candidates[0]
    
            # county-level unique candidate
            if "county" in entity.lower():
    
                state_scores[candidate_state] += 3
    
                debug_info.append({
    
                    "entity_type": "county",
    
                    "entity": entity,
    
                    "state_candidates": state_candidates
                })
    
                # -----------------------------------------
                # user consistency bonus
                # -----------------------------------------
    
                for user_state in user_context:
    
                    if user_state == candidate_state:
    
                        state_scores[user_state] += 1
    
                        debug_info.append({
    
                            "user_location_bonus": user_state
                        })
    
            # -------------------------------------------------
            # POI unique candidate
            # -------------------------------------------------
    
            else:
    
                state_scores[candidate_state] += 2
    
                debug_info.append({
    
                    "entity_type": "poi",
    
                    "entity": entity,
    
                    "state_candidates": state_candidates
                })
    
        # =================================================
        # MULTIPLE CANDIDATES
        # =================================================
    
        else:
    
            # -------------------------------------------------
            # county-level multiple states
            # -------------------------------------------------
    
            if "county" in entity.lower():
    
                for state in state_candidates:
    
                    state_scores[state] += 1
    
                debug_info.append({
    
                    "entity_type": "county",
    
                    "entity": entity,
    
                    "state_candidates": state_candidates
                })
    
                # ---------------------------------------------
                # user consistency bonus
                # ---------------------------------------------
    
                for user_state in user_context:
    
                    if user_state in state_scores:
    
                        state_scores[user_state] += 1
    
                        debug_info.append({
    
                            "user_location_bonus": user_state
                        })
    
            # -------------------------------------------------
            # POI / vague location
            # -------------------------------------------------
    
            else:
    
                constrained_found = False
    
                # ---------------------------------------------
                # entity + user_state retrieval
                # ---------------------------------------------
    
                for user_state in user_context:
    
                    query = f"{entity} {user_state}"
    
                    constrained_candidates = \
                        get_entity_state_candidates(query)
    
                    if len(constrained_candidates) > 0:
    
                        constrained_found = True
    
                        for state in constrained_candidates:
    
                            state_scores[state] += 2
    
                        debug_info.append({
    
                            "entity_type": "poi",
    
                            "query": query,
    
                            "state_candidates":
                                constrained_candidates
                        })
    
                # ---------------------------------------------
                # fallback global retrieval
                # ---------------------------------------------
    
                if not constrained_found:
    
                    for state in state_candidates:
    
                        state_scores[state] += 1
    
                    debug_info.append({
    
                        "fallback_entity": entity,
    
                        "state_candidates": state_candidates
                    })

    # =====================================================
    # MULTI ENTITY
    # =====================================================

    else:

        for entity in entities:

            state_candidates = get_entity_state_candidates(entity)

            if len(state_candidates) == 0:
                continue

            for state in state_candidates:

                state_scores[state] += 1

            debug_info.append({

                "entity": entity,

                "state_candidates": state_candidates
            })

        # -------------------------------------------------
        # user consistency bonus
        # -------------------------------------------------

        for user_state in user_context:

            if user_state in state_scores:

                state_scores[user_state] += 1

                debug_info.append({

                    "user_location_bonus": user_state
                })

    # =====================================================
    # FINAL RESULT
    # =====================================================

    if len(state_scores) == 0:

        return {

            "status": "no_candidate",

            "predicted_state": None,

            "scores": {},

            "debug": debug_info
        }

    sorted_scores = sorted(

        state_scores.items(),

        key=lambda x: x[1],

        reverse=True
    )

    best_state = sorted_scores[0][0]

    return {

        "status": "resolved",

        "predicted_state": best_state,

        "scores": dict(sorted_scores),

        "debug": debug_info
    }
# =========================================================
# PROCESS CSV
# =========================================================

def process_csv():

    df = pd.read_csv(INPUT_CSV)

    results = []

    for idx, row in df.iterrows():

        print("=" * 60)
        print(f"Processing Row {idx}")

        # =================================================
        # POST ID
        # =================================================

        post_id = row.get("id")

        # =================================================
        # LLM ANALYSIS
        # =================================================

        llm_analysis_str = row.get("llm_analysis", "")

        entities = parse_llm_analysis(llm_analysis_str)

        print("Entities:", entities)

        if len(entities) == 0:

            results.append({

                "row_id": idx,

                "post_id": post_id,

                "entities": None,

                "user_location": None,

                "predicted_state": None,

                "status": "no_entity",

                "scores": None,

                "debug": None
            })

            continue

        # =================================================
        # USER LOCATION
        # =================================================

        user_location = None

        try:

            user_data = ast.literal_eval(str(row.get("user", "{}")))

            if isinstance(user_data, dict):

                user_location = user_data.get("location")

        except:
            pass

        print("User Location:", user_location)

        # =================================================
        # USER STATE CANDIDATES
        # =================================================

        user_context = parse_user_location(user_location)

        print("User State Candidates:", user_context)

        # =================================================
        # STATE INFERENCE
        # =================================================

        result = infer_state(

            entities=entities,

            user_context=user_context
        )

        print("Prediction:", result["predicted_state"])

        # =================================================
        # SAVE RESULT
        # =================================================

        results.append({

            "row_id": idx,

            "post_id": post_id,

            "entities": str(entities),

            "user_location": user_location,

            "predicted_state": result["predicted_state"],

            "status": result["status"],

            "scores": json.dumps(result["scores"]),

            "debug": json.dumps(result["debug"])
        })

    # =====================================================
    # SAVE CSV
    # =====================================================

    output_df = pd.DataFrame(results)

    output_df.to_csv(

        OUTPUT_CSV,

        index=False
    )

    print("=" * 60)
    print("DONE")
    print("Saved:", OUTPUT_CSV)

# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    process_csv()

Processing Row 0
Entities: ['Colorado']
User Location: Las Vegas, NV
User State Candidates: ['Nevada']
Prediction: Colorado
Processing Row 1
Entities: ['Paradise Park', 'High Park']
User Location: Colorado
User State Candidates: ['Colorado', 'Texas']
Prediction: North Carolina
Processing Row 2
Entities: ['Cache La Poudre Middle School', 'West County Road 54G', 'Laporte', 'Colorado']
User Location: Denver, Colorado
User State Candidates: ['Colorado']
Prediction: Colorado
Processing Row 3
Entities: ['North Central', 'Northeastern', 'Colorado', 'High Park', 'CO']
User Location: DENVER, CO. 5,280'FT U.S.A.
User State Candidates: []
Prediction: Pennsylvania
Processing Row 4
Entities: ['High Park']
User Location: Lancaster CA
User State Candidates: ['California']
Prediction: California
Processing Row 5
Entities: ['Colorado']
User Location: California
User State Candidates: ['California']
Prediction: Colorado
Processing Row 6
Entities: ['Colorado']
User Location: Nairobi , Kenya
User State Ca